# Initialize Git Repository
https://docs.ycrc.yale.edu/clusters-at-yale/guides/github/

In [ ]:
#git init -b main  # Initializes the repo with 'main' as the default branch
#git add .
#git commit -m "Initial commit"
#git remote add origin git@github.com:innacohen/mod-extract.git
#git push -u origin main --force

# Download mod files as a single JSON file

In [ ]:
import os
import pandas as pd
import requests
import json
import re
from tqdm import tqdm

# Load dataset from Excel
df = pd.read_excel("../data/model_db_annotations.xlsx")

# Output JSON files
output_json = "../data/mod_files.json"
failed_json = "../data/failed_downloads.json"

# List to store mod file data and failed downloads
mod_files_data = []
failed_downloads = []

# Function to convert ModelDB URL to direct download URL
def get_direct_download_url(url):
    match = re.search(r"https://modeldb\.science/(\d+)\?tab=2&file=(.+)", url)
    if match:
        model_id, file_path = match.groups()
        return f"https://modeldb.science/getModelFile?model={model_id}&file={file_path}", file_path
    return None, None  # Return None if the URL doesn't match the expected pattern

# Function to fetch .mod file content and store in JSON
def fetch_mod_file(row_id, file_hash, raw_sha, count, url):
    direct_url, file_path = get_direct_download_url(url)
    
    # Default structure for JSON entry
    file_entry = {
        "row_id": row_id,
        "file_hash": file_hash,
        "raw_sha": raw_sha,
        "count": count,
        "url": url,
        "download_url": direct_url,
        "content": None  # Default to None (indicating missing content)
    }

    if not direct_url:
        print(f"🚨 Skipping invalid URL: {url}")
        failed_downloads.append(file_entry)
        mod_files_data.append(file_entry)  # Store even failed ones
        return

    try:
        response = requests.get(direct_url, timeout=10)
        response.raise_for_status()
        file_entry["content"] = response.text  # Store the downloaded content
    except requests.exceptions.RequestException as e:
        print(f"❌ Failed to fetch {direct_url}: {e}")
        failed_downloads.append(file_entry)  # Log failed download

    mod_files_data.append(file_entry)  # Store the file entry (success or failure)

# Select rows from 467 (corresponding to row_id 468) and take 11 rows
start = 467
n_rows = 20 
df_subset = df.iloc[start:start+n_rows]

# Process selected files
for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="Fetching .mod files"):
    fetch_mod_file(
        row["row_id"], 
        row["file_hash"], 
        row["raw_sha"], 
        row["count"], 
        row["url"]
    )

# Save all .mod data to JSON (including failed downloads)
with open(output_json, "w", encoding="utf-8") as json_file:
    json.dump(mod_files_data, json_file, indent=4)

# Save failed downloads separately for easy retrying
if failed_downloads:
    with open(failed_json, "w", encoding="utf-8") as json_file:
        json.dump(failed_downloads, json_file, indent=4)
    print(f"⚠️ Some downloads failed. Check {failed_json} for details.")

print(f"\n✅ All .mod files (including failures) saved in {output_json}")


# Read JSON file

In [1]:
import os
import pandas as pd
import requests
import json
import re
from bs4 import BeautifulSoup
from tqdm import tqdm

In [2]:
def View(df, rows=5, cols=None, width=None):
    """Displays the first `rows` of the DataFrame like R's View() by adjusting Pandas settings."""
    
    # Show only the first `rows` of the DataFrame
    with pd.option_context(
        "display.max_rows", rows,  # Limit number of rows shown
        "display.max_columns", cols,  # Show all columns
        "display.max_colwidth", width,  # Show full column width
        "display.expand_frame_repr", False  # Prevent column wrapping
    ):
        display(df.head(rows))  # Show only the first `rows`


In [42]:
# Function to extract mod directory from the URL
def extract_dir(url):
    match = re.search(r"file=([^/]+)/[^/]+\.mod", url)  # Extract the directory name before the .mod file
    return match.group(1) if match else None  # Return directory name if found, else None

# Function to extract mod file name without extension
def extract_fname(url):
    match = re.search(r"/([^/]+)\.mod$", url)  # Get filename without extension
    return match.group(1) if match else None  # Return only the name (e.g., 'na')

# Function to extract model_id from the URL
def extract_model_id(url):
    match = re.search(r"https://modeldb\.science/(\d+)", url)
    return int(match.group(1)) if match else None  # Convert to integer

# Function to determine exclusion reason with snake_case labels
def determine_exclusion(row):
    if pd.isna(row["content"]):
        return "not_found"  # Standardized exclusion for missing files
    if "x86_64" in row["url"]:
        return "x86_64"  # Standardized exclusion for architecture issues
    return None  # Keep valid entries as None (not excluded)

# Function to extract all TITLE occurrences from .mod content
def extract_title(content):
    if pd.isna(content):  # If content is None or NaN, return None
        return None
    matches = re.findall(r"^TITLE\s+(.+)", content, re.MULTILINE)  # Find all TITLE lines
    return matches if matches else None  # Return list if multiple, else None

# Function to extract all COMMENT sections from .mod content
def extract_comment(content):
    if pd.isna(content):  # If content is None or NaN, return None
        return None
    matches = re.findall(r"COMMENT\s+(.*?)\s+ENDCOMMENT", content, re.DOTALL)  # Capture all COMMENT sections
    return matches if matches else None  # Return list if multiple, else None

# Function to extract all SUFFIX occurrences from .mod content
def extract_suffix(content):
    if pd.isna(content):  # If content is None or NaN, return None
        return None
    matches = re.findall(r"SUFFIX\s+(\w+)", content)  # Find all SUFFIX statements
    return matches if matches else None  # Return list if multiple, else None

# Function to extract all USEION occurrences from .mod content
def extract_use_ion(content):
    if pd.isna(content):  # If content is None or NaN, return None
        return None
    matches = re.findall(r"USEION\s+(\w+)", content)  # Find all USEION statements
    return matches if matches else None  # Return list if multiple, else None



# Function to extract all ions listed after READ but stopping before WRITE
def extract_read_ion(content):
    if pd.isna(content):  # If content is None or NaN, return None
        return None
    
    matches = re.findall(r"USEION\s+\w+\s+READ\s+([\w,\s]+?)(?:\s+WRITE|\n|$)", content)  # Capture all READ ions
    read_ions = [ion.strip() for match in matches for ion in re.split(r"[,\s]+", match) if ion]  # Flatten & clean list
    
    return read_ions if read_ions else None  # Return list if found, else None

# Function to extract all ions listed after WRITE, stopping at the newline
def extract_write_ion(content):
    if pd.isna(content):  # If content is None or NaN, return None
        return None

    # Find all WRITE statements on their respective lines
    matches = re.findall(r"WRITE\s+([^\n]*)", content)

    if not matches:
        return None

    # Flatten list, remove extra spaces, and split by commas/whitespace
    write_ions = [ion.strip() for match in matches for ion in re.split(r"[,\s]+", match) if ion]

    return write_ions if write_ions else None  # Return cleaned list


# Function to extract all NONSPECIFIC currents
def extract_nonspecific_current(content):
    if pd.isna(content):  # If content is None or NaN, return None
        return None

    # Find all NONSPECIFIC_CURRENT statements and capture their contents
    matches = re.findall(r"NONSPECIFIC_CURRENT\s+([^\n]*)", content)

    if not matches:
        return None

    # Flatten list, remove extra spaces, and split by commas/whitespace
    nonspecific_currents = [curr.strip() for match in matches for curr in re.split(r"[,\s]+", match) if curr]

    return nonspecific_currents if nonspecific_currents else None  # Return cleaned list


# Function to extract RANGE variables (before GLOBAL)
def extract_range(content):
    if pd.isna(content):
        return None

    # Find RANGE block and stop at the next GLOBAL block if present
    match = re.search(r"RANGE\s+([\w\s,]+?)(?=\s+GLOBAL|\s*$)", content, re.MULTILINE)

    if not match:
        return None

    # Extract variables from the RANGE block
    range_vars = [var.strip() for var in match.group(1).split(",") if var.strip()]

    return range_vars if range_vars else None  # Return cleaned list

# Function to extract all GLOBAL variables while removing "GLOBAL" itself
def extract_global(content):
    if pd.isna(content):
        return None

    # Find all GLOBAL blocks, allowing for spaces and newlines
    matches = re.findall(r"GLOBAL\s+([\w\s,]+)", content, re.MULTILINE)

    if not matches:
        return None

    # Join all GLOBAL blocks into a single string, replacing newlines with spaces
    cleaned_text = " ".join(matches).replace("\n", " ")

    # Split on commas and whitespace, ensuring variables are correctly separated
    global_vars = [var.strip() for var in re.split(r"[,\s]+", cleaned_text) if var and var.upper() != "GLOBAL"]

    return global_vars if global_vars else None  # Return cleaned list


# Function to extract parameter names and values as a dictionary
def extract_parameter(content):
    if pd.isna(content):  # If content is None or NaN, return None
        return None
    
    # Regex to find parameter names and values, ignoring units in parentheses
    matches = re.findall(r"(\w+)\s*=\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)", content)  
    
    # Convert to dictionary (float conversion only for valid numbers)
    param_dict = {name: float(value) for name, value in matches if re.match(r"^-?\d*\.?\d+(?:[eE][-+]?\d+)?$", value)}
    
    return param_dict if param_dict else None  # Return dictionary or None if empty

# Function to extract webpage heading
def extract_heading(url):
    try:
        response = requests.get(url, timeout=10)  # Fetch the webpage
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Try extracting heading from the most relevant tag
        heading = soup.find("h1")  # Look for <h1> (main title)
        return heading.text.strip() if heading else None  # Return text or None
    except requests.exceptions.RequestException:
        return None  # Return None if the request fail

# Function to extract citation (text inside parentheses)
def extract_citation(heading):
    if pd.isna(heading):
        return None
    match = re.search(r"\(([^)]+)\)", heading)  # Find text inside parentheses
    return match.group(1) if match else None  # Extract citation


# Function to extract first author(s) (removes "et al." and "al" correctly)
def extract_author(citation):
    if pd.isna(citation):
        return None

    # Extract first author(s) before "et al" or variants
    match = re.search(r"^([\w\s&\-,]+?)(?:\s+et\s+al\.?|et)?(?:,|\s|$)", citation)  
    first_author = match.group(1).strip() if match else None  

    # Remove any trailing "al" left behind
    if first_author:
        first_author = re.sub(r"\b(al)\b", "", first_author, flags=re.IGNORECASE).strip()

    return first_author

# Function to extract the first year from citation (including shortened years like '20)
def extract_year(citation):
    if pd.isna(citation):
        return None
    match = re.search(r"\b(19|20)?\d{2}\b|'\d{2}", citation)  # Find 4-digit or short year ('20)
    if match:
        return match.group(0).replace("'", "")  # Remove apostrophe but keep year as '20' if short
    return None  # Return None if no year found

In [53]:
# Load JSON into DataFrame
df = pd.read_json("../data/mod_files.json")

# Set "row_id" as the index
df.set_index("row_id", inplace=True)

# Exclude mod files
df["mod_exclude"] = df.apply(determine_exclusion, axis=1)  # Apply exclusion function

# Extract features from url
df["mod_heading"] = df["url"].apply(extract_heading)  # Extract heading from webpage
df["mod_citation"] = df["mod_heading"].apply(extract_citation)
df["mod_first_author"] = df["mod_citation"].apply(extract_author)
df["mod_year"] = df["mod_citation"].apply(extract_year)  # Now handles multiple years


df["mod_dir"] = df["url"].apply(extract_dir)
df["mod_name"] = df["url"].apply(extract_fname)
df["mod_model_id"] = df["url"].apply(extract_model_id)

#  Extract features from content
df["mod_title"] = df["content"].apply(extract_title)
df["mod_comment"] = df["content"].apply(extract_comment)
df["mod_suffix"] = df["content"].apply(extract_suffix)
df["mod_use_ion"] = df["content"].apply(extract_use_ion)
df["mod_read_ion"] = df["content"].apply(extract_read_ion)
df["mod_write_ion"] = df["content"].apply(extract_write_ion)
df["mod_nonspecific_current"] = df["content"].apply(extract_nonspecific_current)
df["mod_range"] = df["content"].apply(extract_range)
df["mod_global"] = df["content"].apply(extract_global)
df["mod_parameter"] = df["content"].apply(extract_parameter)
df["mod_state"] = df["content"].apply(extract_state)

In [33]:
#Questions
#Is it okay that we make assumptions like start after READ and stop after WRITE, what if there is no WRITE statement?

In [54]:
df.columns

Index(['file_hash', 'raw_sha', 'count', 'url', 'download_url', 'content',
       'mod_exclude', 'mod_heading', 'mod_citation', 'mod_first_author',
       'mod_year', 'mod_dir', 'mod_name', 'mod_model_id', 'mod_title',
       'mod_comment', 'mod_suffix', 'mod_use_ion', 'mod_read_ion',
       'mod_write_ion', 'mod_nonspecific_current', 'mod_range', 'mod_global',
       'mod_parameter', 'mod_state'],
      dtype='object')

In [55]:
df

,file_hash,raw_sha,count,url,download_url,content,mod_exclude,mod_heading,mod_citation,mod_first_author,...,mod_comment,mod_suffix,mod_use_ion,mod_read_ion,mod_write_ion,mod_nonspecific_current,mod_range,mod_global,mod_parameter,mod_state
row_id,,,,,,,,,,,,,,,,,,,,,
468,25b50b5441b9a5492122214e0648e0529ec055f2d6e97b...,f81e972ce9d730e924c8280e95395b26346853b9c0b397...,3,https://modeldb.science/58173?tab=2&file=b05oc...,https://modeldb.science/getModelFile?model=581...,None,not_found,None,None,None,...,None,None,None,None,None,None,None,None,None,None
469,3fb039ac47d8004f4d9da9bc3da43bf89ce3e9c632e87d...,5a432e65a6d72ef8943f0ae6e833100108f341f793c0a0...,1,https://modeldb.science/116983?tab=2&file=thet...,https://modeldb.science/getModelFile?model=116...,TITLE I-h channel from Magee 1998 for distal d...,None,CA1 pyramidal neuron: h channel-dependent defi...,Marcelin et al. 2008,Marcelin,...,None,[hd],None,None,None,[i],"[ghdbar, vhalfl, tau]","[linf, taua, taud]","{'ghdbar': 0.0001, 'vhalfl': -100.0, 'kl': -12...",[l]
470,338c3525e3a21e4cdc628b3224a949883cf12b2765f872...,e8daa7ec610c947cbfa1bef33e16ba2748e74da7e2596e...,2,https://modeldb.science/136026?tab=2&file=djur...,https://modeldb.science/getModelFile?model=136...,"\nCOMMENT\n\nna.mod\n\nSodium channel, Hodgkin...",None,Functional structure of mitral cell dendritic ...,Djurisic et al. 2008,Djurisic,...,"[na.mod\n\nSodium channel, Hodgkin-Huxley styl...",[na],[na],[ena],[ina],None,"[m, h, gna, gbar, vshift]","[thm1, thm2, qm1, qm2, thi1, thi2, qi, qinf, t...","{'gbar': 258.272, 'vshift': 0.0, 'thm1': -70.3...","[m, h]"
471,2ee9ed06c3f7f53d4c4a8f65c4f7ed7c979b810983fa72...,eeb928941300620229779ed4262c41055cac12f7b2fd02...,1,https://modeldb.science/18502?tab=2&file=spine...,https://modeldb.science/getModelFile?model=185...,\r\nCOMMENT\na synaptic current with NMDA func...,None,Spine fusion and branching affects synaptic re...,"Rusakov et al 1996, 1997",Rusakov,...,[a synaptic current with NMDA function conduct...,None,None,None,None,[i],"[onset, tau1, tau2, Mgetha, gamma, gmax, e, i]",None,"{'g': 0.0, 'onset': 0.0, 'tau1': 80.0, 'tau2':...",None
472,bcd979facda855ab577dc0cc18f1476bf510c7ac6b642e...,c5130c699809a9cc9fcd117e6257f56da565fdd1b282bc...,1,https://modeldb.science/256140?tab=2&file=Luqu...,https://modeldb.science/getModelFile?model=256...,TITLE Purkinje simplified Cell model\r\n\r\nCO...,None,Spike burst-pause dynamics of Purkinje cells r...,Luque et al 2019,Luque,...,[kM channel\r\n \r\n Authors: Ni...,[Purk_gKM],[k],[ek],[ik],None,"[gkmbar, ik, g]",None,"{'gkmbar': 0.001, 'ek': 125.0, 'alp_M': 0.02, ...",[M]
473,96526eb4a8f585ecbb64d59a643b0e83bfdc195f8bba6a...,bed60dff2ab3a09598fe1476f782be2bec39a0b686c866...,1,https://modeldb.science/82894?tab=2&file=nrntr...,https://modeldb.science/getModelFile?model=828...,COMMENT\nFour helpful hints:\n\n1) before call...,None,A single column thalamocortical network model ...,Traub et al 2005,Traub,...,[Four helpful hints:\n\n1) before calling scal...,[nothing],None,None,None,None,None,None,{'x': 0.5},None
474,9bbabc95794782664eff3e6318db36ad8811c38572b48e...,63ca66c9d9f75f25001ff6a2ff622babc7851813152fb7...,1,https://modeldb.science/168858?tab=2&file=Cosk...,https://modeldb.science/getModelFile?model=168...,"TITLE Potasium dr type current for RD Traub, J...",None,Rhesus Monkey Layer 3 Pyramidal Neurons: Young...,Coskren et al. 2015,Coskren,...,[Implemented by Aniruddha Yadav 2007 (aniruddh...,[kdr],[k],[ek],[ik],None,[gbar],None,"{'gbar': 0.0, 'Vkd': 10.0, 'a1': 0.25, 'b1': 4...",[m]
475,20308b254cb6bb8a0e7f2db569dd2078b06b1f874dc1c7...,20308b254cb6bb8a0e7f2db569dd2078b06b1f874dc1c7...,1,https://modeldb.science/249705?tab=2&file=plat...,https://modeldb.science/getModelFile?model=249...,../glutamate.mod,x86_64,Glutamate mediated dendritic and somatic plate...,Gao et al '20,Gao,...,None,None,None,None,None,None,None,None,None,None
476,eaa2b487d69781a92be0b85af13037f3f0036f94971a4c...,1cf926e1537e76053093f7164ebe6383e210d53b184db

In [7]:
View(df.loc[:,["mod_range"]], 10)

,mod_range
row_id,
468,None
469,"[ghdbar, vhalfl, tau]"
470,"[m, h, gna, gbar, vshift]"
471,"[onset, tau1, tau2, Mgetha, gamma, gmax, e, i]"
472,"[gkmbar, ik, g]"
473,None
474,[gbar]
475,None
476,"[r, i]"


In [44]:
! git add .
! git commit -m "added read, write, and state"
! git push 

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date


In [ ]:
!pwd

In [ ]:
View(df,3)

In [39]:
sample = """ 
TITLE FH channel
: Frankenhaeuser - Huxley channels for Xenopus

NEURON {
	SUFFIX fh
	USEION na READ nai, nao WRITE ina
	USEION k READ ki, ko WRITE ik
	NONSPECIFIC_CURRENT il, ip
	RANGE pnabar, pkbar, ppbar, gl, el, il, ip
	GLOBAL inf,tau
}

INCLUDE "standard.inc"
PARAMETER {
	v (mV)
	celsius (degC) : 20
	pnabar=8e-3 (cm/s)
	ppbar=.54e-3 (cm/s)
	pkbar=1.2e-3 (cm/s)
	nai (mM) : 13.74
	nao (mM) : 114.5
	ki (mM) : 120
	ko (mM) : 2.5
	gl=30.3e-3 (mho/cm2)
	el = -69.74 (mV)
}
STATE {
	m h n p
}
ASSIGNED {
	ina (mA/cm2)
	ik (mA/cm2)
	ip (mA/cm2)
	il (mA/cm2)
	inf[4]
	tau[4] (ms)
}

INITIAL {
	mhnp(v*1(/mV))
	m = inf[0]
	h = inf[1]
	n = inf[2]
	p = inf[3]
}

BREAKPOINT {
	LOCAL ghkna
	SOLVE states METHOD cnexp
	ghkna = ghk(v, nai, nao)
	ina = pnabar*m*m*h*ghkna
	ip = ppbar*p*p*ghkna
	ik = pkbar*n*n*ghk(v, ki, ko)
	il = gl*(v - el)
}

FUNCTION ghk(v(mV), ci(mM), co(mM)) (.001 coul/cm3) {
	:assume a single charge
	LOCAL z, eci, eco
	z = (1e-3)*FARADAY*v/(R*(celsius+273.15))
	eco = co*efun(z)
	eci = ci*efun(-z)
	ghk = (.001)*FARADAY*(eci - eco)
}

FUNCTION efun(z) {
	if (fabs(z) < 1e-4) {
		efun = 1 - z/2
	}else{
		efun = z/(exp(z) - 1)
	}
}

DERIVATIVE states {	: exact when v held constant
	mhnp(v*1(/mV))
	m' = (inf[0] - m)/tau[0]
	h' = (inf[1] - h)/tau[1]
	n' = (inf[2] - n)/tau[2]
	p' = (inf[3] - p)/tau[3]
}

UNITSOFF
FUNCTION alp(v(mV),i) { LOCAL a,b,c,q10 :rest = -70  order m,h,n,p
	v = v+70
	q10 = 3^((celsius - 20)/10)
	if (i==0) {
		a=.36 b=22. c=3.
		alp = q10*a*expM1(b - v, c)
	}else if (i==1){
		a=.1 b=-10. c=6.
		alp = q10*a*expM1(v - b, c)
	}else if (i==2){
		a=.02 b= 35. c=10.
		alp = q10*a*expM1(b - v, c)
	}else{
		a=.006 b= 40. c=10.
		alp = q10*a*expM1(b - v , c)
	}
}

FUNCTION bet(v,i) { LOCAL a,b,c,q10 :rest = -70  order m,h,n,p
	v = v+70
	q10 = 3^((celsius - 20)/10)
	if (i==0) {
		a=.4  b= 13.  c=20.
		bet = q10*a*expM1(v - b, c)
	}else if (i==1){
		a=4.5  b= 45.  c=10.
		bet = q10*a/(exp((b - v)/c) + 1)
	}else if (i==2){
		a=.05  b= 10.  c=10.
		bet = q10*a*expM1(v - b, c)
	}else{
		a=.09 b= -25. c=20.
		bet = q10*a*expM1(v - b, c)
	}
}

FUNCTION expM1(x,y) {
	if (fabs(x/y) < 1e-6) {
		expM1 = y*(1 - x/y/2)
	}else{
		expM1 = x/(exp(x/y) - 1)
	}
}

PROCEDURE mhnp(v) {LOCAL a, b :rest = -70
	TABLE inf, tau DEPEND celsius FROM -100 TO 100 WITH 200
	FROM i=0 TO 3 {
		a = alp(v,i)  b=bet(v,i)
		tau[i] = 1/(a + b)
		inf[i] = a/(a + b)
	}
}
UNITSON
"""

In [9]:
sample2 = """
NEURON {
	SUFFIX fh
	USEION na READ nai, nao WRITE ina
	USEION k READ ki, ko WRITE ik
	NONSPECIFIC_CURRENT il, ip
	RANGE pnabar, pkbar, ppbar, gl, el, il, ip
	GLOBAL inf,tau
}
"""

In [51]:
import re

# Function to extract STATE variables
def extract_state(content):
    if pd.isna(content):  # If content is None or NaN, return None
        return None

    # Find all STATE blocks and capture their contents
    matches = re.findall(r"STATE\s*\{([^}]*)\}", content, re.MULTILINE)

    if not matches:
        return None

    # Flatten list, remove extra spaces, and split by commas/whitespace
    state_vars = [state.strip() for match in matches for state in re.split(r"[,\s]+", match) if state]

    return state_vars if state_vars else None  # Return cleaned list


In [52]:
extract_state(sample)

['m', 'h', 'n', 'p']

In [ ]:
! git push